# STATE train/val loss curves — all sizes per dataset

One figure per dataset. Rows = all available sizes (size number shown as row label on the left), each row stacked **train (top) + val (bottom)** on linear axes, with one viridis-colored line per quality.

Pipeline:
1. **Loader** — walk every `{dataset}/{size}/{quality}/results/State/model/loss/metrics.csv` and load the train / val curves.
2. **Plot** — one figure per dataset showing all sizes.

In [54]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path('/home/igor/noise_scaling/data')
DATASETS = ['PBMC', 'larry', 'merfish', 'shendure']
ES_PATIENCE = 5  # state-defaults.yaml: experiment.early_stopping.patience

def _is_size_dir(p: Path) -> bool:
    return p.is_dir() and p.name.isdigit()

def _is_quality_dir(p: Path) -> bool:
    if not p.is_dir():
        return False
    try:
        float(p.name); return True
    except ValueError:
        return False

def _detect_es(val: pd.DataFrame, patience: int) -> tuple[bool, int]:
    if val.empty: return False, 0
    bidx = val['validation/val_loss'].idxmin()
    n_after = len(val) - 1 - bidx
    return (n_after >= patience), n_after

# curves_all[dataset][size][quality] -> dict
curves_all: dict[str, dict[int, dict[float, dict]]] = {}

print(f'{"dataset":>10}  {"size":>9}  {"quality":>10}  {"steps":>7}  {"epochs":>6}  '
      f'{"val_pts":>7}  {"best_step":>9}  {"best_val":>8}  {"after":>5}  {"early_stop":>10}')
print('-' * 105)
for ds in DATASETS:
    ds_dir = DATA_ROOT / ds
    if not ds_dir.is_dir():
        continue
    size_dirs = sorted(filter(_is_size_dir, ds_dir.iterdir()), key=lambda p: int(p.name))
    curves_all[ds] = {}
    for size_dir in size_dirs:
        size = int(size_dir.name)
        qdirs = sorted(filter(_is_quality_dir, size_dir.iterdir()), key=lambda p: float(p.name))
        size_bucket: dict[float, dict] = {}
        for q_dir in qdirs:
            q = float(q_dir.name)
            csv = q_dir / 'results' / 'State' / 'model' / 'loss' / 'metrics.csv'
            if not csv.exists():
                continue
            try:
                df = pd.read_csv(csv, on_bad_lines='skip')
            except Exception:
                continue
            if 'validation/val_loss' not in df.columns or 'trainer/train_loss' not in df.columns:
                continue
            train = df[['step', 'trainer/train_loss']].dropna(subset=['trainer/train_loss']).reset_index(drop=True)
            val   = df[['step', 'validation/val_loss']].dropna(subset=['validation/val_loss']).reset_index(drop=True)
            if val.empty or train.empty:
                continue
            epoch_max = int(df['epoch'].max()) if 'epoch' in df.columns else -1
            bidx = val['validation/val_loss'].idxmin()
            best_step = int(val.iloc[bidx]['step'])
            best_val  = float(val.iloc[bidx]['validation/val_loss'])
            es, n_after = _detect_es(val, ES_PATIENCE)
            size_bucket[q] = {
                'train': train, 'val': val,
                'best_step': best_step, 'best_val': best_val,
                'es_triggered': es, 'n_after_best': n_after,
                'epoch_max': epoch_max,
                'total_steps': int(df['step'].max()) if 'step' in df.columns and not df.empty else 0,
                'csv': csv,
            }
            print(f'{ds:>10}  {size:>9d}  {q:>10g}  {int(df["step"].max()):>7d}  {epoch_max:>6d}  '
                  f'{len(val):>7d}  {best_step:>9d}  {best_val:>8.4f}  {n_after:>5d}  '
                  f'{("YES" if es else "no"):>10}')
        if size_bucket:
            curves_all[ds][size] = size_bucket
    if not curves_all[ds]:
        del curves_all[ds]

total_runs = sum(len(qm) for ds in curves_all for qm in curves_all[ds].values())
total_sizes = sum(len(curves_all[ds]) for ds in curves_all)
print(f'\nLoaded {total_runs} runs across {total_sizes} (dataset, size) combinations in {len(curves_all)} datasets.')

   dataset       size     quality    steps  epochs  val_pts  best_step  best_val  after  early_stop
---------------------------------------------------------------------------------------------------------
      PBMC        100   0.0012346       32      15       32         26   14.4531      5         YES
      PBMC        100   0.0025982       60      29       60         49   14.8280     10         YES
      PBMC        100   0.0054682       60      29       60         52   16.9225      7         YES
      PBMC        100   0.0115083       90      44       90         80   18.4318      9         YES
      PBMC        100     0.02422       76      37       76         65   20.7257     10         YES
      PBMC        100    0.050973       66      32       66         55   18.4555     10         YES
      PBMC        100    0.107277      132      65      132        121   14.9519     10         YES
      PBMC        100    0.225772       82      40       82         80   16.6505      1       

      PBMC      46415           1    15245      20       21      14518   13.6843      1          no
      PBMC     100000   0.0012346    15559       9       10      15003    2.9556      0          no
      PBMC     100000   0.0025982    15619       9       10      15057    6.0949      0          no
      PBMC     100000   0.0054682    15629       9       10      15066   10.1640      0          no
      PBMC     100000   0.0115083    15629       9       10      15066   11.9889      0          no
      PBMC     100000     0.02422    15629       9       10      15066   13.4016      0          no
      PBMC     100000    0.050973    15629       9       10      15066   11.7989      0          no
      PBMC     100000    0.107277    15629       9       10      13503   10.6931      1          no
      PBMC     100000    0.225772    15629       9       10      15066   11.2569      0          no
      PBMC     100000    0.475155    15629       9       10      15066   12.1561      0          no


## One figure per dataset: all sizes × all qualities

For each dataset we render rows = every size that has loss curves on disk.
The size number is the row label on the left; each row stacks **train (top)** and
**val (bottom)** on linear axes, with one viridis-colored line per quality.

In [ ]:
# All sizes per dataset — columns are datasets, rows are size-rank × (train, val).
# Size number is shown as the per-subplot title on the train row (sizes differ
# across datasets at the same rank).  One viridis line per quality.
# Train loss is plotted as a centered rolling mean (window=200 steps) to average
# out minibatch noise; val loss is logged per-epoch and plotted raw.
import matplotlib.pyplot as plt
import numpy as np

TRAIN_SMOOTH_WINDOW = 200  # centered rolling-mean window for train loss (steps)

ds_list = [ds for ds in DATASETS if ds in curves_all and curves_all[ds]]
ncols = len(ds_list)
# Sort each dataset's sizes; rank r uses the r-th smallest size per column.
sizes_by_ds = {ds: sorted(curves_all[ds].keys()) for ds in ds_list}
max_ranks = max(len(sizes_by_ds[ds]) for ds in ds_list)
nrows = 2 * max_ranks

# Taller rows: 2.4 in per subplot (was 1.4); wider tick/label breathing room via hspace.
PER_ROW_H = 2.4
FIG_DPI = 300  # render + save at 300 dpi
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(4.2 * ncols, PER_ROW_H * nrows),
    dpi=FIG_DPI, squeeze=False,
)
cmap = plt.get_cmap('viridis')

# ── Column headers: dataset name at the top of each column ──
for col, ds in enumerate(ds_list):
    axes[0, col].annotate(
        ds, xy=(0.5, 1.35), xycoords='axes fraction',
        ha='center', va='bottom', fontsize=15, fontweight='bold',
    )

# Viridis coloring on row-group labels: smallest rank = dark, largest = bright.
rank_norm = plt.Normalize(0, max_ranks - 1)

for rank in range(max_ranks):
    row_t = 2 * rank
    row_v = row_t + 1

    # ── Row-group label on the far left: rank index (sizes differ per column) ──
    axes[row_t, 0].annotate(
        f'rank {rank + 1}/{max_ranks}',
        xy=(-0.32, -0.05), xycoords='axes fraction',
        ha='center', va='center', fontsize=12, fontweight='bold',
        color=cmap(rank_norm(rank)), rotation=90,
    )

    for col, ds in enumerate(ds_list):
        ax_t, ax_v = axes[row_t, col], axes[row_v, col]
        sizes = sizes_by_ds[ds]

        # Not every dataset has `max_ranks` sizes — render blanks where missing.
        if rank >= len(sizes):
            for ax in (ax_t, ax_v):
                ax.set_xticks([]); ax.set_yticks([])
                for spine in ax.spines.values():
                    spine.set_visible(False)
            continue

        size = sizes[rank]
        qmap = curves_all[ds][size]
        qs = sorted(qmap.keys())
        if not qs:
            for ax in (ax_t, ax_v):
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, color='gray', fontsize=10)
                ax.set_xticks([]); ax.set_yticks([])
            ax_t.set_title(f'N = {size:,}', fontsize=10)
            continue
        norm = (plt.Normalize(np.log10(min(qs)), np.log10(max(qs)))
                if len(qs) > 1 else plt.Normalize(0, 1))
        for q in qs:
            info = qmap[q]
            color = cmap(norm(np.log10(q))) if len(qs) > 1 else cmap(0.5)
            train_pos = info['train'][info['train']['step'] > 0]
            val_pos   = info['val'][info['val']['step'] > 0]
            smooth = train_pos['trainer/train_loss'].rolling(
                window=TRAIN_SMOOTH_WINDOW, min_periods=10, center=True).mean()
            ax_t.plot(train_pos['step'], smooth, color=color, linewidth=1.2, label=f'q={q:g}')
            ax_v.plot(val_pos['step'], val_pos['validation/val_loss'],
                      color=color, marker='o', linewidth=1.4, markersize=4, label=f'q={q:g}')
            ax_v.scatter([info['best_step']], [info['best_val']],
                         color=color, marker='X', s=55, edgecolor='k', linewidth=0.5, zorder=5)
        for ax in (ax_t, ax_v):
            ax.grid(alpha=0.3)
        # Per-subplot title shows the actual size N (sizes differ across columns)
        ax_t.set_title(f'N = {size:,}', fontsize=10)
        ax_v.set_title('')
        if col == 0:
            ax_t.set_ylabel(f'train loss\n(rolling mean, w={TRAIN_SMOOTH_WINDOW})')
            ax_v.set_ylabel('val loss\n(per-epoch, raw)')
        if row_v == nrows - 1:
            ax_v.set_xlabel('optimizer step')
        if col == ncols - 1:
            ax_v.legend(loc='upper right', fontsize=6, ncol=2, title='quality')

plt.suptitle(
    f'STATE loss curves  —  all sizes per dataset  '
    f'(train: centered rolling mean, window={TRAIN_SMOOTH_WINDOW} steps;  val: per-epoch raw)',
    y=1.005, fontsize=13, fontweight='bold',
)
plt.tight_layout()
# Extra vertical spacing between rows now that each subplot is taller.
plt.subplots_adjust(left=0.08, top=0.97, hspace=0.45)
plt.show()

## Smallest vs. largest: side-by-side comparison

For each dataset, show only the **smallest** size that has loss curves and the
**largest** size side-by-side. Columns = {smallest, largest}; within each dataset
row we stack **train (top)** + **val (bottom)**. Viridis color encodes quality.

In [ ]:
# Smallest vs. largest per dataset — columns are datasets, rows are
# (smallest, largest) × (train, val).  Uses `curves_all` from the loader.
# Train loss is plotted as a centered rolling mean (window=200 steps) to average
# out minibatch noise; val loss is logged per-epoch and plotted raw.
import matplotlib.pyplot as plt
import numpy as np

TRAIN_SMOOTH_WINDOW = 200  # centered rolling-mean window for train loss (steps)

ds_list = [ds for ds in DATASETS if ds in curves_all and curves_all[ds]]
ncols = len(ds_list)
categories = ['smallest', 'largest']   # two size groups, each train+val
nrows = 2 * len(categories)            # 4 rows total

FIG_DPI = 300  # render + save at 300 dpi
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(4.2 * ncols, 2.8 * nrows),
    dpi=FIG_DPI, squeeze=False,
)
cmap = plt.get_cmap('viridis')

# ── Column headers: dataset name at the top of each column ──
for col, ds in enumerate(ds_list):
    axes[0, col].annotate(
        ds, xy=(0.5, 1.35), xycoords='axes fraction',
        ha='center', va='bottom', fontsize=15, fontweight='bold',
    )

# ── Row-group labels on the left: SMALLEST / LARGEST, spanning train+val ──
group_colors = {'smallest': '#a33a3a', 'largest': '#1f3b7a'}

for cat_idx, cat_name in enumerate(categories):
    row_t = 2 * cat_idx
    row_v = row_t + 1

    axes[row_t, 0].annotate(
        cat_name.upper(),
        xy=(-0.32, -0.05), xycoords='axes fraction',
        ha='center', va='center', fontsize=14, fontweight='bold',
        color=group_colors[cat_name], rotation=90,
    )

    for col, ds in enumerate(ds_list):
        ax_t, ax_v = axes[row_t, col], axes[row_v, col]
        sizes = sorted(curves_all[ds].keys())
        size = sizes[0] if cat_name == 'smallest' else sizes[-1]
        qmap = curves_all[ds][size]
        qs = sorted(qmap.keys())
        if not qs:
            for ax in (ax_t, ax_v):
                ax.text(0.5, 0.5, 'no data', ha='center', va='center',
                        transform=ax.transAxes, color='gray', fontsize=10)
                ax.set_xticks([]); ax.set_yticks([])
            ax_t.set_title(f'N = {size:,}', fontsize=10)
            continue
        norm = (plt.Normalize(np.log10(min(qs)), np.log10(max(qs)))
                if len(qs) > 1 else plt.Normalize(0, 1))
        for q in qs:
            info = qmap[q]
            color = cmap(norm(np.log10(q))) if len(qs) > 1 else cmap(0.5)
            train_pos = info['train'][info['train']['step'] > 0]
            val_pos   = info['val'][info['val']['step'] > 0]
            smooth = train_pos['trainer/train_loss'].rolling(
                window=TRAIN_SMOOTH_WINDOW, min_periods=10, center=True).mean()
            ax_t.plot(train_pos['step'], smooth, color=color, linewidth=1.2, label=f'q={q:g}')
            ax_v.plot(val_pos['step'], val_pos['validation/val_loss'],
                      color=color, marker='o', linewidth=1.4, markersize=4, label=f'q={q:g}')
            ax_v.scatter([info['best_step']], [info['best_val']],
                         color=color, marker='X', s=55, edgecolor='k', linewidth=0.5, zorder=5)
        for ax in (ax_t, ax_v):
            ax.grid(alpha=0.3)
        # Per-subplot title shows the actual size N for this (dataset, size-group)
        ax_t.set_title(f'N = {size:,}', fontsize=10)
        ax_v.set_title('')
        if col == 0:
            ax_t.set_ylabel(f'train loss\n(rolling mean, w={TRAIN_SMOOTH_WINDOW})')
            ax_v.set_ylabel('val loss\n(per-epoch, raw)')
        if row_v == nrows - 1:
            ax_v.set_xlabel('optimizer step')
        if col == ncols - 1:
            ax_v.legend(loc='upper right', fontsize=6, ncol=2, title='quality')

plt.suptitle(
    f'STATE loss curves  —  smallest vs. largest size per dataset  '
    f'(train: centered rolling mean, window={TRAIN_SMOOTH_WINDOW} steps;  val: per-epoch raw)',
    y=1.02, fontsize=13, fontweight='bold',
)
plt.tight_layout()
plt.subplots_adjust(left=0.08, top=0.94)
plt.show()